In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' can be imported
import sys
from pathlib import Path # for path manipulations
# Move three levels up from current working directory to reach workspace root
workspace_root = Path.cwd().parent.parent.parent.resolve() 
smx_dir = workspace_root / 'smx'  # Path to smx folder
if str(smx_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(smx_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{workspace_root}/real_datasets/xrf/sweet_pepper.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '2.12':'23.08']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '2.12':'23.08'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '2.12':'23.08'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-27 11:10:03,885 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-27 11:10:03,889 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [7]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=1,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()
#plsda_results[0]

## Definição das Zonas Espectrais

In [8]:
spectral_cuts = [
('S', 2.12, 2.4),
('Cl', 2.4, 3.04),
('K ka', 3.0, 3.44),
('K kb + Ca ka', 3.44, 3.84),
('Ca kb', 3.84, 4.14), #
('background1', 4.14, 5.7),
('Mn', 5.7, 6.04),
('Fe ka', 6.04, 6.8), #
('Fe kb', 6.8, 7.3), #
('Cu', 7.3, 8.32),
('Zn', 8.32, 8.84),
('background2', 8.84, 11.56),
('Br ka', 11.56, 12.5),
('Rb ka + Br kb', 12.5, 13.8),
('Sr ka', 13.8, 14.5), #
('Rb kb', 14.5, 15.3), #
('background3', 15.3, 17.0), 
('Rh ka Compton', 17.0, 19.86),
('Rh ka Rayleigh', 19.86, 20.8),
('Rh kb Compton', 20.8, 22.3),
('Rh kb Rayleigh', 22.3, 23.08)
]

## VIP, Regression Coefficients e SHAP (como no original)

In [9]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

In [11]:
shap_global_importance = pd.read_csv('shap_sweet_pepper.csv', sep=';') # loading previously saved shap_unique_df

# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df

FileNotFoundError: [Errno 2] No such file or directory: 'shap_sweet_pepper.csv'

## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [86]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zone 'S': VE = 52.67%, dim = 15 variables
Zone 'Cl': VE = 79.94%, dim = 33 variables
Zone 'K ka': VE = 98.21%, dim = 23 variables
Zone 'K kb + Ca ka': VE = 63.19%, dim = 21 variables
Zone 'background1': VE = 17.34%, dim = 94 variables
Zone 'Mn': VE = 56.93%, dim = 18 variables
Zone 'Fe': VE = 57.33%, dim = 64 variables
Zone 'Cu': VE = 53.40%, dim = 52 variables
Zone 'Zn': VE = 60.08%, dim = 27 variables
Zone 'background2': VE = 68.69%, dim = 137 variables
Zone 'Br ka': VE = 65.01%, dim = 48 variables
Zone 'Rb ka + Br kb': VE = 74.53%, dim = 66 variables
Zone 'background3': VE = 70.02%, dim = 161 variables
Zone 'Rh ka Compton': VE = 88.63%, dim = 146 variables
Zone 'Rh ka Rayleigh': VE = 82.31%, dim = 46 variables
Zone 'Rh kb Compton': VE = 81.75%, dim = 76 variables
Zone 'Rh kb Rayleigh': VE = 69.11%, dim = 40 variables

Scores DataFrame shape: (36, 17)


In [87]:
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred_cont
)

In [88]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 110 | Discarded: 26
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 110 | Discarded: 26
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 113 | Discarded: 23
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 108 | Discarded: 28
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 107 | Discarded: 29
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 106 | Discarded: 30
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 116 | Discarded: 20
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 119 | Discarded: 17
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 112 | Discarded: 24
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 110 | Discarded: 26
Computing Covariance for Predicates
Metric: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 119 | Discarded: 17
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 111 | Discarded: 25
Ba

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Rb ka + Br kb > -0.02,Rb ka + Br kb > -0.02,Rb ka + Br kb > -0.02,Rb ka + Br kb > -0.28
1,Rb ka + Br kb > -0.28,Rh ka Compton <= 0.45,Rb ka + Br kb > -0.12,Rb ka + Br kb > -0.02
2,Rh ka Compton <= 0.45,Rb ka + Br kb > -0.28,Rb ka + Br kb > -0.28,Rh ka Compton <= -0.05
3,Rh ka Compton <= -0.05,Rh ka Compton <= 0.14,Rh ka Compton <= 0.45,Rb ka + Br kb > -0.12
4,Rb ka + Br kb > -0.12,K ka <= 0.20,K ka <= 0.20,Rh ka Compton <= 0.45
...,...,...,...,...
63,NaN,Rh kb Rayleigh > -0.08,Class_B,Br ka > -0.05
64,NaN,Class_A,NaN,Fe > -0.07
65,NaN,Class_B,NaN,background2 > 0.18
66,NaN,NaN,NaN,Class_A


In [89]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Rb ka + Br kb > -0.02,5.512661,Rb ka + Br kb,-0.02,>
1,Rh ka Compton <= 0.45,3.843443,Rh ka Compton,0.45,<=
2,K ka <= 0.20,3.171713,K ka,0.20,<=
3,Cu <= 0.01,1.887485,Cu,0.01,<=
4,Rh ka Rayleigh <= 0.21,1.849225,Rh ka Rayleigh,0.21,<=
5,background3 <= -0.04,1.733370,background3,-0.04,<=
6,Rh kb Compton <= 0.25,1.665174,Rh kb Compton,0.25,<=
7,Br ka <= 0.04,1.198415,Br ka,0.04,<=
8,Cl <= 0.16,1.100757,Cl,0.16,<=
9,Rh kb Rayleigh <= 0.06,1.012714,Rh kb Rayleigh,0.06,<=


In [90]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zone 'S': VE = 57.86%, dim = 15 variables
Zone 'Cl': VE = 86.73%, dim = 33 variables
Zone 'K ka': VE = 99.30%, dim = 23 variables
Zone 'K kb + Ca ka': VE = 73.64%, dim = 21 variables
Zone 'background1': VE = 21.46%, dim = 94 variables
Zone 'Mn': VE = 58.45%, dim = 18 variables
Zone 'Fe': VE = 62.13%, dim = 64 variables
Zone 'Cu': VE = 55.31%, dim = 52 variables
Zone 'Zn': VE = 60.70%, dim = 27 variables
Zone 'background2': VE = 69.36%, dim = 137 variables
Zone 'Br ka': VE = 64.96%, dim = 48 variables
Zone 'Rb ka + Br kb': VE = 79.32%, dim = 66 variables
Zone 'background3': VE = 69.90%, dim = 161 variables
Zone 'Rh ka Compton': VE = 92.92%, dim = 146 variables
Zone 'Rh ka Rayleigh': VE = 85.65%, dim = 46 variables
Zone 'Rh kb Compton': VE = 83.08%, dim = 76 variables
Zone 'Rh kb Rayleigh': VE = 69.80%, dim = 40 variables


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Rb ka + Br kb > -0.02,5.512661,Rb ka + Br kb,-0.02,>,-0.003744,27.0,0.000729,Rb ka + Br kb > -0.003744
1,Rb ka + Br kb > -0.28,4.258175,Rb ka + Br kb,-0.28,>,-0.136523,0.0,0.000198,Rb ka + Br kb > -0.136523
2,Rh ka Compton <= 0.45,3.843443,Rh ka Compton,0.45,<=,0.421114,2.0,0.002152,Rh ka Compton <= 0.421114
3,Rb ka + Br kb > -0.12,3.829498,Rb ka + Br kb,-0.12,>,-0.062361,9.0,0.002081,Rb ka + Br kb > -0.062361
4,K ka <= 0.20,3.171713,K ka,0.20,<=,0.254042,24.0,0.004192,K ka <= 0.254042
...,...,...,...,...,...,...,...,...,...
69,Rh kb Rayleigh > -0.08,0.368798,Rh kb Rayleigh,-0.08,>,-0.029851,25.0,0.002858,Rh kb Rayleigh > -0.029851
70,K ka > 0.44,0.277886,K ka,0.44,>,0.522114,34.0,0.001978,K ka > 0.522114
71,Cl <= -0.04,0.004421,Cl,-0.04,<=,-0.015929,14.0,0.003224,Cl <= -0.015929
72,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [91]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Rb ka + Br kb':
  - Dimensão: 66 variáveis espectrais
  - Range de energias: 12.5 - 13.8
  - Variância explicada pela PC1: 79.32%


In [92]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='regression',
        metric='mean_abs_diff', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 110 | Discarded: 26
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 110 | Discarded: 26
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 113 | Discarded: 23
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 108 | Discarded: 28
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 107 | Discarded: 29
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 106 | Discarded: 30
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 116 | Discarded: 20
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 119 | Discarded: 17
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 112 | Discarded: 24
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 110 | Discarded: 26
PERTURBATION IMPORTANCE FOR PREDICATES
Task type (aim): regression
Perturbation mode: median
Statistics source: full
Metric: mean_abs_diff
Total folds: 10


[Bag_1] Processing 110 predicates...
  Predicate: S > -0.07 (n=22)
    Zone: 15 columns

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Rb ka + Br kb > -0.02,Rb ka + Br kb > -0.02,K ka <= -0.48,Rb ka + Br kb > -0.02
1,Rh ka Compton <= -0.39,Rb ka + Br kb > -0.12,Rb ka + Br kb > -0.02,Rh ka Compton <= -0.39
2,K ka <= -0.48,Rb ka + Br kb > -0.28,K ka > 0.20,Rb ka + Br kb > -0.12
3,Rb ka + Br kb > -0.12,Rh ka Compton > 0.45,Rh ka Compton > 0.14,K ka <= -0.48
4,Rh ka Compton > 0.45,K ka <= -0.48,Rh ka Compton <= -0.39,K ka <= -0.16
...,...,...,...,...
128,Mn <= 0.02,Mn > -0.02,Mn <= -0.03,Mn <= 0.02
129,Mn > -0.02,Mn <= 0.02,Class_A,Mn > -0.02
130,Mn <= -0.03,Mn <= -0.03,Class_B,Mn <= -0.03
131,Class_A,Class_A,NaN,Class_A


In [93]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Rb ka + Br kb > -0.02,0.156021
1,K ka <= -0.48,0.104479
2,Rb ka + Br kb > -0.12,0.101974
3,Rb ka + Br kb > -0.28,0.086479
4,K ka <= -0.16,0.078341
...,...,...
105,Mn <= -0.02,0.000909
106,Mn > -0.05,0.000903
107,Mn > -0.02,0.000859
108,Mn <= 0.02,0.000798


In [94]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Rb ka + Br kb > -0.02,4.903362,Rb ka + Br kb,-0.02,>
1,K ka <= -0.48,3.878167,K ka,-0.48,<=
2,Rh ka Compton <= -0.39,3.863308,Rh ka Compton,-0.39,<=
3,K kb + Ca ka <= -0.14,1.745503,K kb + Ca ka,-0.14,<=
4,Cl <= -0.20,1.472292,Cl,-0.20,<=
5,Rh kb Compton <= -0.20,1.423690,Rh kb Compton,-0.20,<=
6,Rh ka Rayleigh <= -0.15,1.115642,Rh ka Rayleigh,-0.15,<=
7,background3 > 0.19,1.096380,background3,0.19,>
8,Fe > 0.09,0.752956,Fe,0.09,>
9,Cu > 0.11,0.224926,Cu,0.11,>


In [95]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Rb ka + Br kb > -0.02,4.903362,Rb ka + Br kb,-0.02,>,-0.003744,27.0,0.000729,Rb ka + Br kb > -0.003744
1,K ka <= -0.48,3.878167,K ka,-0.48,<=,-0.553004,18.0,0.004092,K ka <= -0.553004
2,Rh ka Compton <= -0.39,3.863308,Rh ka Compton,-0.39,<=,-0.345973,25.0,0.002620,Rh ka Compton <= -0.345973
3,Rb ka + Br kb > -0.12,3.597328,Rb ka + Br kb,-0.12,>,-0.062361,9.0,0.002081,Rb ka + Br kb > -0.062361
4,Rh ka Compton > 0.45,3.165872,Rh ka Compton,0.45,>,0.421114,2.0,0.002152,Rh ka Compton > 0.421114
...,...,...,...,...,...,...,...,...,...
131,Mn <= 0.02,0.002601,Mn,0.02,<=,0.003812,0.0,0.000531,Mn <= 0.003812
132,Mn > -0.02,0.002551,Mn,-0.02,>,-0.003958,18.0,0.000109,Mn > -0.003958
133,Mn <= -0.03,0.000911,Mn,-0.03,<=,-0.005294,5.0,0.000226,Mn <= -0.005294
134,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


In [96]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Rb ka + Br kb':
  - Dimensão: 66 variáveis espectrais
  - Variância explicada pela PC1: 79.32%


In [97]:
# Permutation importance baseado em mudança nas predições do modelo PLS
# Medimos a média da diferença absoluta entre as predições originais e as predições
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Predições base do modelo PLS
baseline_pred = pls_model.predict(Xcalclass_prep)
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_pred = pls_model.predict(X_perm)
        diffs.append(np.mean(np.abs(baseline_pred - perm_pred)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance': importance_list
})
permutation_df.sort_values(by='Permutation_importance', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance', ascending=False)
permutation_unique_df

,energy,Permutation_importance,Zone
0,13.34,0.015082,Rb ka + Br kb
1,3.24,0.010099,K ka
2,3.68,0.009469,K kb + Ca ka
3,2.58,0.006944,Cl
4,19.32,0.004321,Rh ka Compton
5,16,0.002150,background3
6,21.34,0.002100,Rh kb Compton
7,20.1,0.002022,Rh ka Rayleigh
8,4,0.001969,background1
9,11.88,0.001910,Br ka


In [98]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'VIP_Score': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_Coefficient' : pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,VIP_Score,Reg_Coefficient,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Rb ka + Br kb,Rb ka + Br kb,Rb ka + Br kb,Rb ka + Br kb,Rb ka + Br kb,Rb ka + Br kb
1,K kb + Ca ka,K kb + Ca ka,K ka,K ka,K ka,Rh ka Compton
2,K ka,Cl,Cl,K kb + Ca ka,Rh ka Compton,K ka
3,Rh ka Compton,background1,K kb + Ca ka,Cl,K kb + Ca ka,Cu
4,Cl,background3,Rh ka Compton,Rh ka Compton,Cl,Rh ka Rayleigh
5,background1,K ka,background3,background3,Rh kb Compton,background3
6,Cu,Zn,Rh ka Rayleigh,Rh kb Compton,Rh ka Rayleigh,Rh kb Compton
7,background3,Cu,Rh kb Compton,Rh ka Rayleigh,background3,Br ka
8,background2,Rh ka Compton,Rh kb Rayleigh,background1,Fe,Cl
9,Zn,Rh kb Rayleigh,Br ka,Br ka,Cu,Rh kb Rayleigh


In [99]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_15524\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
9,Shap,Permutation,0.914578
10,Shap,LRC_perturbation,0.879239
12,Permutation,LRC_perturbation,0.879239
0,VIP_Score,Reg_Coefficient,0.817782
2,VIP_Score,Permutation,0.808888
3,VIP_Score,LRC_perturbation,0.783692
14,LRC_perturbation,LRC_covariance,0.764480
1,VIP_Score,Shap,0.756756
6,Reg_Coefficient,Permutation,0.727988
5,Reg_Coefficient,Shap,0.726066


In [100]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')